In [6]:
import yfinance as yf
import pandas as pd

egx_tickers = [
    'COMI.CA', 'ORWE.CA', 'MCRO.CA', 'EKHO.CA', 'HRHO.CA',
    'JUFO.CA', 'ABUK.CA', 'ESRS.CA', 'TMGH.CA', 'SWDY.CA',
    'PHDC.CA', 'OCDI.CA', 'ACGC.CA', 'ETEL.CA', 'EFIC.CA',
]

all_dfs = []

for ticker in egx_tickers:
    try:
        df_tick = yf.download(ticker, start='2018-01-01',
                              auto_adjust=True, progress=False)

        if len(df_tick) < 200:
            print(f"Skipping {ticker} — only {len(df_tick)} rows")
            continue

        # Flatten any multi-level columns
        df_tick.columns = [col[0] if isinstance(col, tuple) else col 
                           for col in df_tick.columns]

        df_tick = df_tick.reset_index()

        # Select only what we need — one stock at a time
        df_tick = df_tick[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()
        df_tick['ticker'] = ticker
        df_tick['Date'] = pd.to_datetime(df_tick['Date'])

        all_dfs.append(df_tick)
        print(f"{ticker}: {len(df_tick)} rows")

    except Exception as e:
        print(f"Failed {ticker}: {e}")

# Stack vertically — each stock is its own rows
df_combined = pd.concat(all_dfs, axis=0, ignore_index=True)

print(f"\nTotal rows: {len(df_combined)}")
print(f"Columns: {df_combined.columns.tolist()}")
print(f"\nSample:")
print(df_combined.head(3))
print(df_combined.tail(3))

COMI.CA: 2056 rows
ORWE.CA: 2056 rows
MCRO.CA: 520 rows
EKHO.CA: 2057 rows
HRHO.CA: 2058 rows
JUFO.CA: 2055 rows
ABUK.CA: 2055 rows
ESRS.CA: 1780 rows
TMGH.CA: 2055 rows
SWDY.CA: 2055 rows
PHDC.CA: 2056 rows
OCDI.CA: 2058 rows
ACGC.CA: 2055 rows
ETEL.CA: 2055 rows
EFIC.CA: 2055 rows

Total rows: 29026
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'ticker']

Sample:
        Date       Open       High        Low      Close   Volume   ticker
0 2018-01-01  26.211632  26.211632  26.211632  26.211632        0  COMI.CA
1 2018-01-02  26.211632  26.211632  26.211632  26.211632        0  COMI.CA
2 2018-01-03  26.086247  25.991364  25.415283  25.533888  1113900  COMI.CA
            Date        Open        High         Low       Close  Volume  \
29023 2026-05-04  213.279999  214.970001  202.100006  206.009995   57979   
29024 2026-05-05  206.009995  214.899994  211.000000  214.600006   12856   
29025 2026-05-06  214.600006  214.899994  211.899994  213.210007   59883   

        ticke

In [8]:
# ── Step 2: Compute features per stock ─────────────────────────────────
def compute_features(df):
    df = df.copy().sort_values('Date').reset_index(drop=True)

    df['return_1d']     = df['Close'].pct_change()
    df['return_5d']     = df['Close'].pct_change(5)
    df['return_10d']    = df['Close'].pct_change(10)
    df['ma_ratio_10']   = df['Close'] / df['Close'].rolling(10).mean()
    df['ma_ratio_50']   = df['Close'] / df['Close'].rolling(50).mean()
    df['volume_ratio']  = df['Volume'] / df['Volume'].rolling(10).mean()

    delta = df['Close'].diff()
    gain  = delta.where(delta > 0, 0).rolling(14).mean()
    loss  = (-delta.where(delta < 0, 0)).rolling(14).mean()
    df['rsi']          = 100 - (100 / (1 + gain / loss))
    df['rsi_momentum'] = df['rsi'] - df['rsi'].shift(3)

    ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['macd']        = ema_12 - ema_26
    df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_hist']   = df['macd'] - df['macd_signal']

    df['volatility_10'] = df['Close'].pct_change().rolling(10).std()
    df['volatility_20'] = df['Close'].pct_change().rolling(20).std()
    df['vol_ratio']     = df['volatility_10'] / df['volatility_20']
    df['day_of_week']   = pd.to_datetime(df['Date']).dt.dayofweek

    df['target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

    return df.dropna()

df_featured = (
    df_combined
    .groupby('ticker', group_keys=False)
    .apply(compute_features)
    .reset_index(drop=True)
)

print(f"Total rows after features: {len(df_featured)}")
print(f"\nRows per stock:")
print(df_featured.groupby('ticker').size().sort_values(ascending=False))

# ── Step 3: Time-aware split by date ───────────────────────────────────
df_featured['Date'] = pd.to_datetime(df_featured['Date'])
cutoff_date = df_featured['Date'].quantile(0.80)
print(f"\nTrain/test cutoff date: {cutoff_date.date()}")

feature_cols = [
    'return_1d', 'return_5d', 'return_10d',
    'ma_ratio_10', 'ma_ratio_50',
    'rsi', 'macd', 'macd_signal', 'macd_hist',
    'volatility_10', 'volatility_20', 'volume_ratio',
    'day_of_week', 'vol_ratio', 'rsi_momentum'
]

train = df_featured[df_featured['Date'] <= cutoff_date]
test  = df_featured[df_featured['Date'] >  cutoff_date]

X_train = train[feature_cols]
y_train = train['target']
X_test  = test[feature_cols]
y_test  = test['target']

print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")
print(f"\nClass balance (train):")
print(y_train.value_counts(normalize=True).round(2))

# ── Step 4: Scale ───────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=feature_cols)
X_test_scaled  = pd.DataFrame(
    scaler.transform(X_test),      columns=feature_cols)

# ── Step 5: Train ───────────────────────────────────────────────────────
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

best_params = {
    'learning_rate': 0.05,
    'max_depth':     5,
    'n_estimators':  300,
    'subsample':     1.0
}

model_combined = XGBClassifier(
    **best_params, random_state=42, eval_metric='logloss')
model_combined.fit(X_train_scaled, y_train)

y_pred = model_combined.predict(X_test_scaled)

print("\n── Combined Model Results ──")
print(f"  Accuracy:  {round(accuracy_score(y_test, y_pred), 3)}")
print(f"  Precision: {round(precision_score(y_test, y_pred), 3)}")
print(f"  Recall:    {round(recall_score(y_test, y_pred), 3)}")
print(f"  F1:        {round(f1_score(y_test, y_pred), 3)}")

cm = confusion_matrix(y_test, y_pred)
print(f"\n  True Down  (0→0): {cm[0,0]}  |  False Up  (0→1): {cm[0,1]}")
print(f"  False Down (1→0): {cm[1,0]}  |  True Up   (1→1): {cm[1,1]}")

print(f"\n  Baseline:         {max(y_test.mean(), 1-y_test.mean()):.3f}")
print(f"  Single stock F1:  0.477")
print(f"  Combined F1:      {round(f1_score(y_test, y_pred), 3)}")

C:\Users\TCS\AppData\Local\Temp\ipykernel_12524\330658195.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_features)


Total rows after features: 28192

Rows per stock:
ticker
HRHO.CA    2009
OCDI.CA    2009
COMI.CA    2007
ORWE.CA    2007
PHDC.CA    2007
ABUK.CA    2006
ACGC.CA    2006
SWDY.CA    2006
TMGH.CA    2006
ETEL.CA    2006
EFIC.CA    2006
JUFO.CA    2005
EKHO.CA    1910
ESRS.CA    1731
MCRO.CA     471
dtype: int64

Train/test cutoff date: 2024-09-16
Train: 22564 rows | Test: 5628 rows

Class balance (train):
target
0    0.56
1    0.44
Name: proportion, dtype: float64

── Combined Model Results ──
  Accuracy:  0.532
  Precision: 0.475
  Recall:    0.278
  F1:        0.351

  True Down  (0→0): 2284  |  False Up  (0→1): 785
  False Down (1→0): 1848  |  True Up   (1→1): 711

  Baseline:         0.545
  Single stock F1:  0.477
  Combined F1:      0.351


In [9]:
# ── Check F1 per stock in test set ─────────────────────────────────────
from sklearn.metrics import f1_score

test['predicted'] = y_pred.values if hasattr(y_pred, 'values') else y_pred
test = test.copy()
test['predicted'] = model_combined.predict(X_test_scaled)

print("F1 per stock in test set:")
print("─" * 35)

stock_results = []
for ticker, group in test.groupby('ticker'):
    if len(group) < 20:
        continue
    idx = group.index
    f1  = f1_score(group['target'], group['predicted'])
    stock_results.append({'Ticker': ticker, 'Rows': len(group), 'F1': round(f1, 3)})

stock_df = pd.DataFrame(stock_results).sort_values('F1', ascending=False)
print(stock_df.to_string(index=False))
print(f"\nMean F1 across stocks: {stock_df['F1'].mean():.3f}")
print(f"Best stock F1:         {stock_df['F1'].max():.3f}")
print(f"Worst stock F1:        {stock_df['F1'].min():.3f}")

F1 per stock in test set:
───────────────────────────────────
 Ticker  Rows    F1
ESRS.CA   125 0.460
COMI.CA   400 0.432
SWDY.CA   400 0.414
JUFO.CA   400 0.405
TMGH.CA   400 0.404
OCDI.CA   400 0.392
HRHO.CA   400 0.377
ABUK.CA   400 0.356
EFIC.CA   400 0.353
ETEL.CA   400 0.301
PHDC.CA   400 0.281
ORWE.CA   400 0.280
MCRO.CA   400 0.276
ACGC.CA   400 0.267
EKHO.CA   303 0.113

Mean F1 across stocks: 0.341
Best stock F1:         0.460
Worst stock F1:        0.113


C:\Users\TCS\AppData\Local\Temp\ipykernel_12524\3874211969.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['predicted'] = y_pred.values if hasattr(y_pred, 'values') else y_pred
